In [0]:

# Run this cell only if you want to remove old files

dbutils.fs.rm(RAW_FOLDER, recurse=True)
dbutils.fs.mkdirs(RAW_FOLDER)

dbutils.fs.rm(LANDING_FOLDER, recurse=True)
dbutils.fs.mkdirs(LANDING_FOLDER)

True

In [0]:
# Step 0 - Generate Raw UPI Data

# In this notebook we create our own UPI transaction data.
# Since no dataset is provided, we generate sample records
# and save them into a Unity Catalog Volume.

# The generated data also contains some bad records like:
# - Missing values
# - Extra spaces
# - Negative amounts
# - Duplicate rows
# - Different timestamp formats

# We also add a few suspicious transactions
# so that we can detect them later.

CATALOG = "main"
SCHEMA = "project_sentinel"
VOLUME = "upi_raw_data"

RAW_FOLDER = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/raw_files"

NUM_FILES = 5
RECORDS_PER_FILE = 500

print(f"Raw data will be written to: {RAW_FOLDER}")

# Create schema and volume if they are not already created.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

dbutils.fs.mkdirs(RAW_FOLDER)

# Functions to generate sample transaction data.

import random
import uuid
import csv
import io
from datetime import datetime, timedelta

random.seed()  

MERCHANT_CATEGORIES = [
    "Grocery", "Fuel", "Electronics", "Travel", "Food_Delivery",
    "P2P_Transfer", "Bill_Payment", "Shopping", "Entertainment", "Other"
]

STATUSES = ["SUCCESS", "SUCCESS", "SUCCESS", "SUCCESS", "FAILED", "PENDING"]

BANKS = ["oksbi", "okhdfcbank", "okicici", "okaxis", "ybl", "paytm"]

#  normal users
NORMAL_USERS = [f"user{n}@{random.choice(BANKS)}" for n in range(1, 121)]

# fraud users
FRAUD_USERS = [f"fraudster{n}@{random.choice(BANKS)}" for n in range(1, 6)]

# a couple of "flagged" IPs that fraud traffic will come from
SUSPICIOUS_IPS = ["45.13.23.9", "103.87.12.201", "185.220.101.4"]


def random_ip(suspicious=False):
    if suspicious:
        return random.choice(SUSPICIOUS_IPS)
    return f"{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(0,255)}"


def random_timestamp(base_time, fmt_choice=0):
    ts = base_time + timedelta(seconds=random.randint(0, 24 * 3600 - 1))
    # inject a couple of different formats to simulate messiness
    if fmt_choice == 0:
        return ts.strftime("%Y-%m-%d %H:%M:%S")
    elif fmt_choice == 1:
        return ts.strftime("%d/%m/%Y %H:%M")
    else:
        return ts.strftime("%Y-%m-%dT%H:%M:%S")


def make_normal_row(base_time):
    ts_fmt = random.choices([0, 1, 2], weights=[85, 10, 5])[0]
    amount = round(random.uniform(10, 5000), 2)

    row = {
        "transaction_id": str(uuid.uuid4()),
        "timestamp": random_timestamp(base_time, ts_fmt),
        "sender_upi": random.choice(NORMAL_USERS),
        "receiver_upi": random.choice(NORMAL_USERS),
        "amount": amount,
        "ip_address": random_ip(suspicious=False),
        "device_id": f"DEV{random.randint(1000,9999)}",
        "status": random.choice(STATUSES),
        "merchant_category": random.choice(MERCHANT_CATEGORIES),
    }
# Introduce a few bad records intentionally.
# This helps demonstrate cleaning in the Silver layer.
    r = random.random()
    if r < 0.03:
        row["amount"] = None                     
    elif r < 0.06:
        row["amount"] = -abs(amount)              
    elif r < 0.09:
        row["amount"] = f"Rs.{amount}"            
    elif r < 0.12:
        row["sender_upi"] = f"  {row['sender_upi']}   "   
    elif r < 0.15:
        row["device_id"] = None                   
    elif r < 0.17:
        row["receiver_upi"] = ""                  

    return row


def make_high_amount_fraud_row(base_time):
    """A single, unusually large transaction from a suspicious IP."""
    row = make_normal_row(base_time)
    row["sender_upi"] = random.choice(NORMAL_USERS)  # looks like a normal user
    row["amount"] = round(random.uniform(150000, 500000), 2)
    row["ip_address"] = random_ip(suspicious=True)
    row["status"] = "SUCCESS"
    return row


def make_velocity_fraud_burst(base_time):
    """Same sender firing many transactions within a few seconds."""
    sender = random.choice(FRAUD_USERS)
    burst_start = base_time + timedelta(seconds=random.randint(0, 24 * 3600 - 1))
    rows = []
    for i in range(random.randint(6, 10)):
        ts = burst_start + timedelta(seconds=i * random.randint(2, 15))
        rows.append({
            "transaction_id": str(uuid.uuid4()),
            "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
            "sender_upi": sender,
            "receiver_upi": random.choice(NORMAL_USERS),
            "amount": round(random.uniform(500, 9000), 2),
            "ip_address": random_ip(suspicious=random.random() < 0.5),
            "device_id": f"DEV{random.randint(1000,9999)}",
            "status": "SUCCESS",
            "merchant_category": "P2P_Transfer",
        })
    return rows


FIELDNAMES = [
    "transaction_id", "timestamp", "sender_upi", "receiver_upi",
    "amount", "ip_address", "device_id", "status", "merchant_category"
]


def generate_file(file_index):
    # anchor to MIDNIGHT of a random recent day (not "now") - combined with
    # the 24-hour random offset in random_timestamp(), this guarantees an
    # even spread across all hours of the day on every run, regardless of
    # what time it actually is when you run this notebook.
    midnight_today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    base_date = midnight_today - timedelta(days=random.randint(0, 3))
    transactions = []

    for _ in range(RECORDS_PER_FILE - 20):
        transactions.append(make_normal_row(base_date))

    #Generate some of high amount fraud transactions
    for _ in range(6):
        transactions.append(make_high_amount_fraud_row(base_date))

     # Generate a few rapid transactions from the same sender.
    #These should later be identified using simple fraud rules.
    for _ in range(random.randint(1, 2)):
        transactions.extend(make_velocity_fraud_burst(base_date))

    # Generate few exact duplicate rows
    if len(transactions) > 5:
        for _ in range(3):
            transactions.append(dict(random.choice(transactions)))

    random.shuffle(transactions)

    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=FIELDNAMES)
    writer.writeheader()
    for transaction in transactions:
        writer.writerow(transaction)

    file_name = f"upi_txns_batch_{file_index}_{int(datetime.now().timestamp())}.csv"
    file_path = f"{RAW_FOLDER}/{file_name}"

    dbutils.fs.put(file_path, buffer.getvalue(), overwrite=True)
    return file_path, len(transactions)


# Generate and save the CSV files

total_rows = 0

for file_num in range(1, NUM_FILES + 1):
    file_path, record_count = generate_file(file_num)
    total_rows += record_count
    print(f"Generated: {file_path}")
    print(f"Records : {record_count}\n")

print("========== Data Generation Complete ==========")
print(f"Files Generated : {NUM_FILES}")
print(f"Total Records   : {total_rows}")
print(f"Saved To        : {RAW_FOLDER}")





Raw data will be written to: /Volumes/main/project_sentinel/upi_raw_data/raw_files
Wrote 66783 bytes.
Generated: /Volumes/main/project_sentinel/upi_raw_data/raw_files/upi_txns_batch_1_1783838838.csv
Records : 497

Wrote 67094 bytes.
Generated: /Volumes/main/project_sentinel/upi_raw_data/raw_files/upi_txns_batch_2_1783838838.csv
Records : 499

Wrote 67718 bytes.
Generated: /Volumes/main/project_sentinel/upi_raw_data/raw_files/upi_txns_batch_3_1783838839.csv
Records : 505

Wrote 66570 bytes.
Generated: /Volumes/main/project_sentinel/upi_raw_data/raw_files/upi_txns_batch_4_1783838839.csv
Records : 496

Wrote 67871 bytes.
Generated: /Volumes/main/project_sentinel/upi_raw_data/raw_files/upi_txns_batch_5_1783838839.csv
Records : 504

========== Data Generation Complete ==========
Files Generated : 5
Total Records   : 2501
Saved To        : /Volumes/main/project_sentinel/upi_raw_data/raw_files


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Step 1 - Raw to Landing
# MAGIC
# MAGIC Copy the raw CSV files into the Landing folder.
# MAGIC No changes are made to the data in this step.

LANDING_FOLDER = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/landing"
dbutils.fs.mkdirs(LANDING_FOLDER)

csv_files = [file for file in dbutils.fs.ls(RAW_FOLDER) if file.name.endswith(".csv")]

if not csv_files:
    print("No csv files found.")
else:
    files_copied = 0
    for file_info in csv_files:
        destination = f"{LANDING_FOLDER}/{file_info.name}"
        dbutils.fs.cp(file_info.path, destination)
        files_copied += 1
        print(f"Copied {file_info.name} /")

    print(f"Files Copied : {files_copied}")
    print(f"Location     : {LANDING_FOLDER}")



Copied upi_txns_batch_1_1783838838.csv /
Copied upi_txns_batch_2_1783838838.csv /
Copied upi_txns_batch_3_1783838839.csv /
Copied upi_txns_batch_4_1783838839.csv /
Copied upi_txns_batch_5_1783838839.csv /
Files Copied : 5
Location     : /Volumes/main/project_sentinel/upi_raw_data/landing


In [0]:
# Step 2 - Landing to Bronze

#Read the CSV files from the Landing folder and store them in a Bronze Delta table.
#No cleaning is done in this step.


BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_upi_transactions"

from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType

bronze_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("sender_upi", StringType(), True),
    StructField("receiver_upi", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("merchant_category", StringType(), True),
])

landing_df = (
    spark.read
    .option("header", True)
    .schema(bronze_schema)
    .csv(LANDING_FOLDER)
)

print(f"Read {landing_df.count()} rows from landing.")

# Add simple metadata columns

bronze_df = (
    landing_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

display(bronze_df.limit(10))

# Write to the Bronze Delta table (append)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table ready: {BRONZE_TABLE}")
print(f"Total rows : {spark.table(BRONZE_TABLE).count()}")



Read 2501 rows from landing.


transaction_id,timestamp,sender_upi,receiver_upi,amount,ip_address,device_id,status,merchant_category,ingestion_timestamp,source_file
da8fbc4f-4762-4022-9eae-7243c05ff2b7,2026-07-12 09:31:22,fraudster5@oksbi,user18@oksbi,3479.24,64.57.29.201,DEV8086,SUCCESS,P2P_Transfer,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
eef49af8-2401-471e-b626-d015d8a7afac,2026-07-12 18:53:38,user9@okhdfcbank,user24@okaxis,3868.79,174.6.190.147,DEV5531,SUCCESS,Food_Delivery,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
5e6612e9-f27e-43bf-acf5-5e47cd98c830,2026-07-12 08:05:37,user3@oksbi,user93@okicici,4622.61,75.120.36.77,DEV3408,SUCCESS,Shopping,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
31a250fa-7545-4e31-8149-9fe96af4ab57,2026-07-12 06:52:00,user69@paytm,user37@okicici,449.62,171.138.149.135,DEV5710,SUCCESS,P2P_Transfer,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
810a8e66-40c7-4985-a1d9-b7ddd936bea3,2026-07-12 07:09:38,fraudster4@okaxis,user91@okaxis,3767.25,204.33.96.59,DEV2653,SUCCESS,P2P_Transfer,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
8c40dd8f-07c3-4f2d-be2d-111347d90efc,2026-07-12 04:46:28,user53@ybl,user34@okicici,4289.86,103.160.7.180,DEV1080,FAILED,Food_Delivery,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
f150f048-7f68-4d37-b915-583538601166,12/07/2026 19:23,user87@okicici,user91@okaxis,655.99,62.134.217.134,DEV9046,SUCCESS,Fuel,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
819e716c-cb4c-404b-ab1f-f0cf3ace7ad7,2026-07-12 19:07:23,user60@ybl,user48@okaxis,4544.32,36.74.106.5,DEV3272,SUCCESS,Shopping,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
1cf19c0d-7e35-4d44-83cc-a35a646bef1a,2026-07-12 11:33:33,user89@oksbi,user92@ybl,null,170.198.22.105,DEV8675,SUCCESS,Fuel,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv
044a73b9-6916-40f6-bd40-d90cffafa5df,2026-07-12 22:33:54,user74@okhdfcbank,user95@okhdfcbank,4560.59,194.50.75.148,DEV3568,SUCCESS,Other,2026-07-12T06:47:43.856Z,dbfs:/Volumes/main/project_sentinel/upi_raw_data/landing/upi_txns_batch_5_1783838839.csv


Bronze table ready: main.project_sentinel.bronze_upi_transactions
Total rows : 30040


In [0]:
# Step 3 - Bronze to Silver
# This is where we actually clean the data:
# trim whitespace
# mask PII 
# handle null values and type mismatches.
# filter out invalid records (negative amounts)

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_upi_transactions"


bronze_df = spark.table(BRONZE_TABLE)
print(f"Starting with {bronze_df.count()} bronze rows.")

#Remove whitespace and turn empty strings into real nulls


from pyspark.sql.functions import col, trim, when, length

string_cols = ["transaction_id", "sender_upi", "receiver_upi",
               "ip_address", "device_id", "status", "merchant_category"]

cleaned_df = bronze_df
for c in string_cols:
    cleaned_df = cleaned_df.withColumn(c, trim(col(c)))
    cleaned_df = cleaned_df.withColumn(c, when(length(col(c)) == 0, None).otherwise(col(c)))

#Fix the amount column (remove currency symbols, cast to double)
# Extract the numeric value from the amount column.

from pyspark.sql.functions import regexp_extract, expr

amount_number_pattern = r"(-?\d+\.?\d*)"

amount_df = cleaned_df.withColumn("amount_extracted",regexp_extract(trim(col("amount")), amount_number_pattern, 1)).withColumn("amount_clean",expr("try_cast(amount_extracted as double)"))

# Format the timestamp column

from pyspark.sql.functions import coalesce, try_to_timestamp, lit

timestamp_df = amount_df.withColumn(
    "event_timestamp",
    coalesce(
        try_to_timestamp(col("timestamp"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("timestamp"), lit("dd/MM/yyyy HH:mm")),
        try_to_timestamp(col("timestamp"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))

# Drop invalid rows 
clean_df = (
    timestamp_df
    .filter(col("transaction_id").isNotNull())
    .filter(col("sender_upi").isNotNull())
    .filter(col("receiver_upi").isNotNull())
    .filter(col("event_timestamp").isNotNull())
    .filter(col("amount_clean").isNotNull())
    .filter(col("amount_clean") > 0)          
)

print(f"Rows remaining after filtering invalid records: {clean_df.count()}")

#Remove duplicate transactions

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

unique_window = Window.partitionBy("transaction_id").orderBy(desc("ingestion_timestamp"))

unique_df = (
    clean_df
    .withColumn("row_num", row_number().over(unique_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(f"Rows remaining after de-duplication: {unique_df.count()}")

# Mask PII

from pyspark.sql.functions import concat, lit, substring, split, element_at, sha2

def mask_upi(colname):
    handle = element_at(split(col(colname), "@"), 1)
    domain = element_at(split(col(colname), "@"), 2)
    hashed_handle = substring(sha2(handle, 256), 1, 8)
    return concat(lit("usr_"), hashed_handle, lit("@"), domain)

masked_df = (
    unique_df
    .withColumn("sender_upi_masked", mask_upi("sender_upi"))
    .withColumn("receiver_upi_masked", mask_upi("receiver_upi"))
)

# the clean Silver columns

silver_df = masked_df.select(
    col("transaction_id"),
    col("event_timestamp").alias("txn_timestamp"),
    col("sender_upi_masked").alias("sender_upi"),
    col("receiver_upi_masked").alias("receiver_upi"),
    col("amount_clean").alias("amount"),
    col("ip_address"),
    col("device_id"),
    col("status"),
    col("merchant_category"),
    col("ingestion_timestamp"),
)

display(silver_df.limit(10))

# Write the Silver Delta table

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table ready: {SILVER_TABLE}")
print(f"Total clean rows: {spark.table(SILVER_TABLE).count()}")




Starting with 30040 bronze rows.
Rows remaining after filtering invalid records: 27611
Rows remaining after de-duplication: 9166


transaction_id,txn_timestamp,sender_upi,receiver_upi,amount,ip_address,device_id,status,merchant_category,ingestion_timestamp
00074ead-2f31-4b01-80c4-35ab5a4a36fe,2026-07-11T21:47:08.000Z,usr_6025d18f@ybl,usr_ddd1c00c@paytm,4865.22,23.127.208.153,DEV2226,FAILED,P2P_Transfer,2026-07-12T06:07:15.295Z
0010404f-4bcd-4049-b02c-9e2882cdf540,2026-07-11T20:51:50.000Z,usr_a36b4f42@paytm,usr_47d59d7a@okaxis,3174.42,107.33.127.99,DEV5271,FAILED,Travel,2026-07-12T06:07:15.295Z
00104c3e-45b3-4caf-8ff9-beb271531031,2026-07-09T17:19:53.000Z,usr_d4af0a3e@okicici,usr_581afbf1@okaxis,1658.46,196.252.115.23,DEV3220,SUCCESS,Fuel,2026-07-12T06:07:15.295Z
00201b89-eb15-4a9a-bb43-ac965abaf1cf,2026-07-10T18:25:01.000Z,usr_fe1f9f0f@oksbi,usr_b6caf3d2@ybl,4703.11,130.195.99.52,DEV7426,SUCCESS,Fuel,2026-07-12T06:07:15.295Z
002e54fc-1f9d-4408-a3d3-98c698764928,2026-07-10T22:45:27.000Z,usr_f8beac41@paytm,usr_4de41535@okhdfcbank,2751.39,53.56.111.51,DEV4731,SUCCESS,Travel,2026-07-12T06:07:15.295Z
003dee9a-84e3-4156-9f67-0c784712ed9b,2026-07-09T23:55:30.000Z,usr_ddd1c00c@paytm,usr_47a268f4@okaxis,1979.13,204.147.218.152,DEV8991,SUCCESS,Food_Delivery,2026-07-12T06:07:15.295Z
0040ddbb-2a97-4918-a624-3b5c01b2b83b,2026-07-10T14:45:00.000Z,usr_ce063ef2@okicici,usr_2bc6ebdf@okhdfcbank,669.85,121.86.93.214,DEV4129,SUCCESS,Bill_Payment,2026-07-12T06:47:45.783Z
00416213-6140-4806-b151-883aa01e773f,2026-07-10T20:19:21.000Z,usr_fe1f9f0f@oksbi,usr_46b5b8fd@ybl,3208.99,78.212.5.153,DEV8962,SUCCESS,Shopping,2026-07-12T06:07:15.295Z
004676b8-59cd-4e6e-9a5b-bed490a7b016,2026-07-12T21:24:31.000Z,usr_0a041b94@okicici,usr_4d4ec949@okicici,2009.9,140.146.161.145,DEV8882,SUCCESS,Electronics,2026-07-12T06:47:45.783Z
0061a57c-48e7-4b40-add7-d1a0220e20bd,2026-07-10T11:44:01.000Z,usr_9c10f999@okhdfcbank,usr_4a8f0735@okaxis,4064.83,127.54.68.20,DEV2412,SUCCESS,Food_Delivery,2026-07-12T06:07:15.295Z


Silver table ready: main.project_sentinel.silver_upi_transactions
Total clean rows: 9166


In [0]:
# Step 4 - Silver to Gold
# This is the final layer. We use the clean Silver data to:
# Build a simple KPI summary table (business metrics)
# Run a simple rule-based fraud scoring engine
# Save flagged transactions into a fraud alerts table

GOLD_KPI_TABLE = f"{CATALOG}.{SCHEMA}.gold_kpi_daily_summary"
GOLD_ALERTS_TABLE = f"{CATALOG}.{SCHEMA}.gold_fraud_alerts"
GOLD_ALL_TRANSACTIONS_TABLE = f"{CATALOG}.{SCHEMA}.gold_transactions_scored"

silver_df = spark.table(SILVER_TABLE)
print(f"Working with {silver_df.count()} clean transactions.")

# Business KPIs

from pyspark.sql.functions import (col, to_date, count, sum as _sum, avg, when, round as _round)

daily_df = silver_df.withColumn("txn_date", to_date(col("txn_timestamp")))

kpi_df = (daily_df.groupBy("txn_date")
          .agg(count("*").alias("total_transactions"),
                _round(_sum("amount"), 2).alias("total_volume"),
                _round(avg("amount"), 2).alias("avg_transaction_value"),
                count(when(col("status") == "SUCCESS", True)).alias("success_count"),
                count(when(col("status") == "FAILED", True)).alias("failed_count"),
                count(when(col("status") == "PENDING", True)).alias("pending_count"),)
          .withColumn("success_rate_pct",_round(col("success_count") * 100.0 / col("total_transactions"), 2))
          .orderBy("txn_date"))

print("Daily KPI summary:")
display(kpi_df)

#Top categories by transaction volume

top_categories_df = (silver_df.groupBy("merchant_category").agg(
        count("*").alias("total_transactions"),
        _round(_sum("amount"), 2).alias("total_volume"),)
        .orderBy(col("total_volume").desc()))

display(top_categories_df)


# Fraud Scoring 
# High Amount rule --> amount > 100000 |score = +50 
# Velocity rule --> same sender makes 5+ transactions within 5 minutes |score = +40 
# Odd Hour rule --> transaction between 1AM-4AM and amount > 50000 |score = +20 
# score >= 70 -> HIGH, 40-69 -> MEDIUM, 1-39 -> LOW

HIGH_AMOUNT_THRESHOLD = 100000
VELOCITY_WINDOW_MINUTES = 5
VELOCITY_TXN_THRESHOLD = 5
ODD_HOUR_AMOUNT_THRESHOLD = 50000

# Rule High amount

scored_df = silver_df.withColumn(
    "flag_high_amount", col("amount") > HIGH_AMOUNT_THRESHOLD
)


# Rule Velocity (count of transactions per sender)

from pyspark.sql.window import Window
from pyspark.sql.functions import unix_timestamp, count as _count

velocity_window = (
    Window.partitionBy("sender_upi")
    .orderBy(unix_timestamp(col("txn_timestamp")))
    .rangeBetween(-VELOCITY_WINDOW_MINUTES * 60, 0)
)

scored_df = scored_df.withColumn("txns_in_window", _count("transaction_id").over(velocity_window)).withColumn("flag_velocity", col("txns_in_window") >= VELOCITY_TXN_THRESHOLD)


# Rule 3: Odd hour + large amount

from pyspark.sql.functions import hour

scored_df = scored_df.withColumn(
    "flag_odd_hour",
    (hour(col("txn_timestamp")).between(1, 4)) & (col("amount") > ODD_HOUR_AMOUNT_THRESHOLD)
)

# Combine rules into a fraud score 

scored_df = (
    scored_df
    .withColumn(
        "fraud_score",
        when(col("flag_high_amount"), 50).otherwise(0) +
        when(col("flag_velocity"), 40).otherwise(0) +
        when(col("flag_odd_hour"), 20).otherwise(0)
    )
    .withColumn(
        "fraud_severity",
        when(col("fraud_score") >= 70, "HIGH")
        .when(col("fraud_score") >= 40, "MEDIUM")
        .when(col("fraud_score") > 0, "LOW")
        .otherwise("NONE")
    )
    .withColumn(
        "is_fraud", col("fraud_score") > 0   # simple True/False flag, easy to filter on
    )
)

#reasons column
from pyspark.sql.functions import concat_ws, array_remove, array

scored_df = scored_df.withColumn(
    "fraud_reasons",
    concat_ws(
        ", ",
        array_remove(
            array(
                when(col("flag_high_amount"), "HIGH_AMOUNT").otherwise(""),
                when(col("flag_velocity"), "VELOCITY").otherwise(""),
                when(col("flag_odd_hour"), "ODD_HOUR_LARGE_AMOUNT").otherwise(""),
            ),
            ""
        )
    )
)

fraud_alerts_df = (
    scored_df
    .filter(col("fraud_score") > 0)
    .select(
        "transaction_id", "txn_timestamp", "sender_upi", "receiver_upi",
        "amount", "ip_address", "merchant_category",
        "is_fraud", "fraud_score", "fraud_severity", "fraud_reasons",
    )
    .orderBy(col("fraud_score").desc())
)

print(f"Flagged {fraud_alerts_df.count()} suspicious transaction(s) out of {silver_df.count()}.")
display(fraud_alerts_df)

# All Transactions with Fraud Status

all_transactions_scored_df = scored_df.select(
    "transaction_id", "txn_timestamp", "sender_upi", "receiver_upi",
    "amount", "ip_address", "merchant_category", "status",
    "is_fraud", "fraud_score", "fraud_severity", "fraud_reasons",
).orderBy(col("txn_timestamp").desc())

display(all_transactions_scored_df.limit(20))

# Write the Gold tables

(
    kpi_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_KPI_TABLE)
)

(
    fraud_alerts_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ALERTS_TABLE)
)

(
    all_transactions_scored_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ALL_TRANSACTIONS_TABLE)
)

print(f"Gold KPI table ready: {GOLD_KPI_TABLE}")
print(f"Gold fraud alerts table ready: {GOLD_ALERTS_TABLE}")
print(f"Gold all-transactions (with is_fraud flag) table ready: {GOLD_ALL_TRANSACTIONS_TABLE}")

#If any HIGH severity fraud found,alert shows

high_severity_rows = fraud_alerts_df.filter(col("fraud_severity") == "HIGH").collect()
medium_severity_count = fraud_alerts_df.filter(col("fraud_severity") == "MEDIUM").count()
low_severity_count = fraud_alerts_df.filter(col("fraud_severity") == "LOW").count()

if len(high_severity_rows) > 0:
    print("=" * 70)
    print("🚨  FRAUD ALERT - HIGH SEVERITY TRANSACTIONS DETECTED  🚨")
    print("=" * 70)
    print(f"{len(high_severity_rows)} HIGH severity transaction(s) found.")
    print(f"(also {medium_severity_count} MEDIUM and {low_severity_count} LOW severity)")
    print("-" * 70)
    for row in high_severity_rows[:10]:
        print(
            f"Txn: {row['transaction_id'][:8]}...  | "
            f"Sender: {row['sender_upi']:<15} | "
            f"Amount: Rs.{row['amount']:>10,.2f} | "
            f"IP: {row['ip_address']:<15} | "
            f"Reasons: {row['fraud_reasons']}"
        )
    print("=" * 70)

else:
    print("No HIGH severity fraud detected in this batch.")
    if medium_severity_count > 0 or low_severity_count > 0:
        print(f"({medium_severity_count} MEDIUM, {low_severity_count} LOW severity transactions were flagged fo review)")


Working with 9166 clean transactions.
Daily KPI summary:


txn_date,total_transactions,total_volume,avg_transaction_value,success_count,failed_count,pending_count,success_rate_pct
2026-07-09,4144,2.785996368E7,6722.96,2792,667,685,67.37
2026-07-10,2729,1.700695178E7,6231.94,1862,405,462,68.23
2026-07-11,1827,1.278416914E7,6997.36,1209,306,312,66.17
2026-07-12,466,2703490.59,5801.48,322,75,69,69.1


merchant_category,total_transactions,total_volume
P2P_Transfer,1121,7128531.74
Bill_Payment,904,6547445.42
Other,892,6533939.25
Food_Delivery,926,6516632.66
Fuel,871,6312266.52
Shopping,936,5908361.11
Electronics,800,5602796.16
Grocery,906,5577354.66
Entertainment,940,5504854.5
Travel,870,4722393.17


Flagged 235 suspicious transaction(s) out of 9166.


transaction_id,txn_timestamp,sender_upi,receiver_upi,amount,ip_address,merchant_category,is_fraud,fraud_score,fraud_severity,fraud_reasons
cf9f6bf7-0903-42e6-a463-e3a43814a762,2026-07-09T01:21:05.000Z,usr_6e5acacc@okaxis,usr_cd8cbd30@oksbi,244885.75,185.220.101.4,P2P_Transfer,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
f548fa14-8dcd-452a-8956-d8a8b6a94037,2026-07-09T02:29:57.000Z,usr_7febe54e@okhdfcbank,usr_bbc790b0@okicici,371826.19,185.220.101.4,Food_Delivery,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
67e62a2a-3d8b-45e5-84e7-825b9f6486e9,2026-07-10T03:34:23.000Z,usr_f1240fde@okaxis,usr_2bc6ebdf@okhdfcbank,294797.87,45.13.23.9,Entertainment,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
e3756588-1f20-4aa9-a6e5-4c4f364fe6d7,2026-07-10T01:24:40.000Z,usr_81115e31@okhdfcbank,usr_ebc835d1@ybl,476657.93,185.220.101.4,Entertainment,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
d0d15735-5463-4ad8-a44b-78cce2ec2d43,2026-07-10T02:36:28.000Z,usr_39a6bd68@ybl,usr_daf7996f@okaxis,241900.43,185.220.101.4,Grocery,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
e8012255-4a9d-4db1-b418-c1077c1703dd,2026-07-11T04:15:31.000Z,usr_659d6055@okicici,usr_f81e0105@okhdfcbank,483590.66,45.13.23.9,Fuel,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
7dd522bc-823d-4eeb-b84d-fabec7f3b42a,2026-07-09T01:56:13.000Z,usr_06f01542@okicici,usr_604f489f@okaxis,325890.51,103.87.12.201,Bill_Payment,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
c774b832-758d-48ab-873a-0d932478879f,2026-07-10T02:01:15.000Z,usr_023736a4@okicici,usr_06f01542@okicici,250174.68,45.13.23.9,Bill_Payment,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
f297c7d7-1965-4a18-a1a0-4fb2d5266230,2026-07-10T04:39:26.000Z,usr_47d59d7a@okhdfcbank,usr_896b1da4@oksbi,241676.39,45.13.23.9,Grocery,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"
38fe0683-82ef-4f58-bb78-89c962bbd230,2026-07-11T03:12:21.000Z,usr_d4af0a3e@ybl,usr_8a43bbd1@paytm,382683.11,45.13.23.9,Electronics,true,70,HIGH,"HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT"


transaction_id,txn_timestamp,sender_upi,receiver_upi,amount,ip_address,merchant_category,status,is_fraud,fraud_score,fraud_severity,fraud_reasons
7a75123f-4459-44f1-ba80-0c04001fed1f,2026-07-12T23:58:55.000Z,usr_f1240fde@okaxis,usr_da08eac8@okhdfcbank,2459.14,223.10.209.170,Grocery,SUCCESS,false,0,NONE,
cf7ce578-54c5-4ea1-9da2-fa0d977d1225,2026-07-12T23:56:26.000Z,usr_9de6a56e@okhdfcbank,usr_62752906@oksbi,2494.48,155.229.155.106,P2P_Transfer,SUCCESS,false,0,NONE,
ca68450c-8a22-4673-a39b-0963f366438b,2026-07-12T23:55:44.000Z,usr_604f489f@okhdfcbank,usr_97cdd97c@oksbi,3533.27,139.40.229.188,Electronics,PENDING,false,0,NONE,
944590e8-bfba-4e89-b2f7-a4eb775eb4b7,2026-07-12T23:55:42.000Z,usr_fb44d98b@okicici,usr_dd6d3ece@okicici,1145.08,172.76.246.199,Other,PENDING,false,0,NONE,
f64303a7-4f0b-4c9f-9945-6c82f90d5f72,2026-07-12T23:54:13.000Z,usr_746465eb@okaxis,usr_81115e31@oksbi,2471.8,223.214.52.174,Other,FAILED,false,0,NONE,
b257cd39-0bcb-4f86-ba33-59935a8d061b,2026-07-12T23:52:21.000Z,usr_407d3c1d@okaxis,usr_4a8f0735@okhdfcbank,221.47,162.34.103.155,Entertainment,FAILED,false,0,NONE,
406b787a-4237-40e1-9c62-8bba289359af,2026-07-12T23:48:52.000Z,usr_8fab3a60@paytm,usr_27a27107@paytm,1433.92,61.193.50.146,Grocery,PENDING,false,0,NONE,
c0615688-79a3-42fe-b21e-2a013bceda1f,2026-07-12T23:45:54.000Z,usr_45f1afd8@oksbi,usr_06f01542@okhdfcbank,3841.3,7.209.177.96,Food_Delivery,SUCCESS,false,0,NONE,
81b1122a-5b40-4f0f-bd2c-b61f07affe75,2026-07-12T23:43:45.000Z,usr_8a43bbd1@paytm,usr_611f51d4@okaxis,3780.2,126.67.241.207,Bill_Payment,SUCCESS,false,0,NONE,
222f05ad-aeee-418f-8122-c4d13ca157e2,2026-07-12T23:37:27.000Z,usr_b917c707@okicici,usr_38da5d67@okicici,324662.61,103.87.12.201,Electronics,SUCCESS,true,50,MEDIUM,HIGH_AMOUNT


Gold KPI table ready: main.project_sentinel.gold_kpi_daily_summary
Gold fraud alerts table ready: main.project_sentinel.gold_fraud_alerts
Gold all-transactions (with is_fraud flag) table ready: main.project_sentinel.gold_transactions_scored
🚨  FRAUD ALERT - HIGH SEVERITY TRANSACTIONS DETECTED  🚨
16 HIGH severity transaction(s) found.
(also 219 MEDIUM and 0 LOW severity)
----------------------------------------------------------------------
Txn: c774b832...  | Sender: usr_023736a4@okicici | Amount: Rs.250,174.68 | IP: 45.13.23.9      | Reasons: HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT
Txn: 7dd522bc...  | Sender: usr_06f01542@okicici | Amount: Rs.325,890.51 | IP: 103.87.12.201   | Reasons: HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT
Txn: dbde5b1d...  | Sender: usr_2bc6ebdf@paytm | Amount: Rs.244,508.98 | IP: 103.87.12.201   | Reasons: HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUNT
Txn: d0d15735...  | Sender: usr_39a6bd68@ybl | Amount: Rs.241,900.43 | IP: 185.220.101.4   | Reasons: HIGH_AMOUNT, ODD_HOUR_LARGE_AMOUN